# Amazon ML Challenge — Business Entity Resolution
## Phase 1: Dataset Profiling

This notebook profiles the training and test datasets before building normalization, blocking, candidate generation, and entity matching models.

### Objectives
- Inspect dataset structure
- Count records
- Analyze file sizes and system resources
- Analyze missing values
- Analyze country distribution
- Analyze duplicate entity IDs
- Analyze business-name and address characteristics
- Analyze ground-truth match distribution
- Analyze S2/S3 match proportions
- Inspect multilingual and noisy text characteristics
- Inspect examples from the training data


In [1]:
import os
import gc
import sys
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Platform:", platform.platform())


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas: 2.3.3
NumPy: 2.0.2
Platform: Linux-6.12.90+-x86_64-with-glibc2.35


In [2]:
# Kaggle dataset path
BASE = Path("/kaggle/input/datasets/bhushanssh/ml-challange/student_resource")

TRAIN = BASE / "dataset" / "train"
TEST = BASE / "dataset" / "test"
UTILS = BASE / "utils"

# Training
train_s1 = TRAIN / "train_source1.tsv"
train_s2 = TRAIN / "train_source2.tsv"
train_s3 = TRAIN / "train_source3.tsv"
ground_truth = TRAIN / "train_ground_truth.tsv"

# Test
test_s1 = TEST / "test_source1.tsv"
test_s2 = TEST / "test_source2.tsv"
test_s3 = TEST / "test_source3.tsv"

files = {
    "train_source1": train_s1,
    "train_source2": train_s2,
    "train_source3": train_s3,
    "ground_truth": ground_truth,
    "test_source1": test_s1,
    "test_source2": test_s2,
    "test_source3": test_s3,
}

print("Base:", BASE)
print("\nFile existence:")
for name, path in files.items():
    print(f"{name:20s}: {path.exists()}")


Base: /kaggle/input/datasets/bhushanssh/ml-challange/student_resource

File existence:
train_source1       : True
train_source2       : True
train_source3       : True
ground_truth        : True
test_source1        : True
test_source2        : True
test_source3        : True


In [3]:
print("DATASET STRUCTURE")
print("=" * 70)

for root, dirs, filenames in os.walk(BASE):
    level = len(Path(root).relative_to(BASE).parts)
    indent = "    " * level
    print(f"{indent}{Path(root).name}/")
    for filename in sorted(filenames):
        file_path = Path(root) / filename
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"{indent}    {filename}  ({size_mb:.2f} MB)")


DATASET STRUCTURE
student_resource/
    Documentation_template.md  (0.00 MB)
    README.md  (0.01 MB)
    utils/
        validate_submission.py  (0.01 MB)
    dataset/
        test/
            test_source1.tsv  (166.91 MB)
            test_source2.tsv  (485.86 MB)
            test_source3.tsv  (482.56 MB)
        train/
            train_ground_truth.tsv  (121.13 MB)
            train_source1.tsv  (200.34 MB)
            train_source2.tsv  (466.63 MB)
            train_source3.tsv  (480.37 MB)


In [4]:
print("FILE SIZES")
print("=" * 70)

file_size_data = []

for name, path in files.items():
    size_bytes = path.stat().st_size
    file_size_data.append({
        "file": name,
        "size_MB": round(size_bytes / (1024 ** 2), 2),
        "size_GB": round(size_bytes / (1024 ** 3), 3),
    })

file_sizes_df = pd.DataFrame(file_size_data)
display(file_sizes_df.sort_values("size_MB", ascending=False).reset_index(drop=True))

print("Total:", round(file_sizes_df["size_MB"].sum(), 2), "MB")


FILE SIZES


,file,size_MB,size_GB
0,test_source2,485.86,0.474
1,test_source3,482.56,0.471
2,train_source3,480.37,0.469
3,train_source2,466.63,0.456
4,train_source1,200.34,0.196
5,test_source1,166.91,0.163
6,ground_truth,121.13,0.118


Total: 2403.8 MB


In [5]:
import psutil

print("SYSTEM RESOURCES")
print("=" * 70)

print("CPU cores:", os.cpu_count())

ram = psutil.virtual_memory()
print("Total RAM:", round(ram.total / (1024 ** 3), 2), "GB")
print("Available RAM:", round(ram.available / (1024 ** 3), 2), "GB")
print("Used RAM:", round(ram.used / (1024 ** 3), 2), "GB")
print("RAM usage:", ram.percent, "%")


SYSTEM RESOURCES
CPU cores: 4
Total RAM: 31.35 GB
Available RAM: 30.05 GB
Used RAM: 0.85 GB
RAM usage: 4.1 %


In [6]:
def count_lines(path):
    with open(path, "rb") as f:
        return sum(1 for _ in f) - 1

row_counts = {}

print("ROW COUNTS")
print("=" * 70)

for name, path in files.items():
    rows = count_lines(path)
    row_counts[name] = rows
    print(f"{name:20s}: {rows:>12,}")


ROW COUNTS
train_source1       :    2,206,821
train_source2       :    5,034,616
train_source3       :    5,285,603
ground_truth        :    2,206,821
test_source1        :    1,732,544
test_source2        :    4,887,273
test_source3        :    5,082,316


In [7]:
summary_df = pd.DataFrame({
    "dataset": list(row_counts.keys()),
    "rows": list(row_counts.values()),
})

summary_df["file_size_MB"] = [
    files[name].stat().st_size / (1024 ** 2)
    for name in summary_df["dataset"]
]
summary_df["file_size_MB"] = summary_df["file_size_MB"].round(2)

display(summary_df)


,dataset,rows,file_size_MB
0,train_source1,2206821,200.34
1,train_source2,5034616,466.63
2,train_source3,5285603,480.37
3,ground_truth,2206821,121.13
4,test_source1,1732544,166.91
5,test_source2,4887273,485.86
6,test_source3,5082316,482.56


In [8]:
print("SCHEMA + SAMPLE RECORDS")
print("=" * 70)

for name, path in files.items():
    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")

    df_sample = pd.read_csv(path, sep="\t", nrows=5)

    print("Columns:")
    print(list(df_sample.columns))

    print("\nData types:")
    print(df_sample.dtypes)

    print("\nSample:")
    display(df_sample)

    del df_sample


SCHEMA + SAMPLE RECORDS

train_source1
Columns:
['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Sample:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India



train_source2
Columns:
['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Sample:


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



train_source3
Columns:
['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Sample:


...

,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine",France
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar, Awas Vikas Colony",India
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Rue Pierre Dignac",France



test_source2
Columns:
['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Sample:


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karnataka",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US



test_source3
Columns:
['entity_id', 'business_name', 'business_address', 'country']

Data types:
entity_id           object
business_name       object
business_address    object
country             object
dtype: object

Sample:


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V, Gurugram, Gurgaon, HR",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, null, A-68, दिल्ली",India


In [9]:
expected_source_columns = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]

expected_gt_columns = [
    "source1_entity_id",
    "matched_entity_ids"
]

print("SCHEMA VALIDATION")
print("=" * 70)

for name, path in files.items():
    df = pd.read_csv(path, sep="\t", nrows=1)

    expected = expected_gt_columns if name == "ground_truth" else expected_source_columns
    actual = list(df.columns)

    status = actual == expected

    print(f"{name:20s}:", "PASS" if status else "CHECK")

    if not status:
        print("  Expected:", expected)
        print("  Actual:  ", actual)

    del df


SCHEMA VALIDATION
train_source1       : PASS
train_source2       : PASS
train_source3       : PASS
ground_truth        : PASS
test_source1        : PASS
test_source2        : PASS
test_source3        : PASS


In [10]:
def profile_missing_values(path, chunksize=200_000):
    total_rows = 0
    missing_counts = Counter()

    for chunk in pd.read_csv(path, sep="\t", chunksize=chunksize):
        total_rows += len(chunk)

        for col in chunk.columns:
            missing_counts[col] += int(chunk[col].isna().sum())

    records = []
    for col, count in missing_counts.items():
        records.append({
            "column": col,
            "missing": count,
            "missing_pct": round(count / total_rows * 100, 3)
        })

    return pd.DataFrame(records)

missing_profiles = {}

for name, path in files.items():
    print(f"\nProfiling missing values: {name}")
    profile = profile_missing_values(path)
    missing_profiles[name] = profile
    display(profile)



Profiling missing values: train_source1


,column,missing,missing_pct
0,entity_id,0,0.0
1,business_name,0,0.0
2,business_address,0,0.0
3,country,0,0.0



Profiling missing values: train_source2


,column,missing,missing_pct
0,entity_id,0,0.000
1,business_name,2,0.000
2,business_address,168967,3.356
3,country,0,0.000



Profiling missing values: train_source3


...

,column,missing,missing_pct
0,entity_id,0,0.0
1,business_name,0,0.0
2,business_address,0,0.0
3,country,0,0.0



Profiling missing values: test_source2


,column,missing,missing_pct
0,entity_id,0,0.000
1,business_name,46,0.001
2,business_address,129408,2.648
3,country,0,0.000



Profiling missing values: test_source3


,column,missing,missing_pct
0,entity_id,0,0.000
1,business_name,59,0.001
2,business_address,136098,2.678
3,country,0,0.000


In [11]:
def profile_countries(path, chunksize=200_000):
    counter = Counter()
    total = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["country"],
        chunksize=chunksize
    ):
        values = (
            chunk["country"]
            .fillna("<MISSING>")
            .value_counts()
            .to_dict()
        )
        counter.update(values)
        total += len(chunk)

    result = []
    for country, count in counter.most_common():
        result.append({
            "country": country,
            "records": count,
            "percentage": round(count / total * 100, 3)
        })

    return pd.DataFrame(result)

country_profiles = {}

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3"
]:
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    country_profiles[name] = profile_countries(files[name])
    display(country_profiles[name])



train_source1


,country,records,percentage
0,US,1323633,59.979
1,India,883188,40.021



train_source2


,country,records,percentage
0,US,3016817,59.921
1,India,2017799,40.079



train_source3


...

,country,records,percentage
0,India,809986,46.751
1,US,663106,38.274
2,France,259452,14.975



test_source2


,country,records,percentage
0,India,2312565,47.318
1,US,1871330,38.290
2,France,703378,14.392



test_source3


,country,records,percentage
0,India,2405000,47.321
1,US,1945701,38.284
2,France,731615,14.395


In [12]:
def find_duplicate_ids(path, chunksize=300_000):
    all_ids = set()
    duplicate_count = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        chunksize=chunksize
    ):
        for entity_id in chunk["entity_id"].dropna():
            if entity_id in all_ids:
                duplicate_count += 1
            else:
                all_ids.add(entity_id)

    return duplicate_count, len(all_ids)

print("DUPLICATE ENTITY IDs")
print("=" * 70)

duplicate_results = []

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3"
]:
    duplicates, unique_ids = find_duplicate_ids(files[name])

    duplicate_results.append({
        "dataset": name,
        "unique_ids": unique_ids,
        "duplicate_ids": duplicates
    })

    print(
        f"{name:20s} | "
        f"unique: {unique_ids:,} | "
        f"duplicates: {duplicates:,}"
    )

duplicate_df = pd.DataFrame(duplicate_results)
display(duplicate_df)


DUPLICATE ENTITY IDs
train_source1        | unique: 2,206,821 | duplicates: 0
train_source2        | unique: 5,034,616 | duplicates: 0
train_source3        | unique: 5,285,603 | duplicates: 0
test_source1         | unique: 1,732,544 | duplicates: 0
test_source2         | unique: 4,887,273 | duplicates: 0
test_source3         | unique: 5,082,316 | duplicates: 0


,dataset,unique_ids,duplicate_ids
0,train_source1,2206821,0
1,train_source2,5034616,0
2,train_source3,5285603,0
3,test_source1,1732544,0
4,test_source2,4887273,0
5,test_source3,5082316,0


In [13]:
gt_sample = pd.read_csv(
    ground_truth,
    sep="\t",
    nrows=5
)

print("GROUND TRUTH")
print("=" * 70)
print("Columns:")
print(list(gt_sample.columns))
print("\nSample:")
display(gt_sample)

del gt_sample


GROUND TRUTH
Columns:
['source1_entity_id', 'matched_entity_ids']

Sample:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


In [14]:
def profile_ground_truth(path, chunksize=200_000):
    match_count_distribution = Counter()
    source_match_distribution = Counter()

    total_rows = 0
    total_matches = 0

    for chunk in pd.read_csv(path, sep="\t", chunksize=chunksize):
        total_rows += len(chunk)

        values = chunk["matched_entity_ids"].fillna("")

        for value in values:
            value = str(value).strip()

            if not value:
                match_count_distribution[0] += 1
                continue

            ids = [x.strip() for x in value.split(",") if x.strip()]
            match_count = len(ids)

            match_count_distribution[match_count] += 1
            total_matches += match_count

            for entity_id in ids:
                if entity_id.startswith("S2-"):
                    source_match_distribution["S2"] += 1
                elif entity_id.startswith("S3-"):
                    source_match_distribution["S3"] += 1
                else:
                    source_match_distribution["OTHER"] += 1

    return (
        total_rows,
        total_matches,
        match_count_distribution,
        source_match_distribution
    )

gt_rows, total_matches, match_distribution, source_distribution = profile_ground_truth(
    ground_truth
)

print("GROUND TRUTH PROFILE")
print("=" * 70)
print("S1 entities:", f"{gt_rows:,}")
print("Total S2/S3 matches:", f"{total_matches:,}")
print("Average matches per S1:", round(total_matches / gt_rows, 4))


GROUND TRUTH PROFILE
S1 entities: 2,206,821
Total S2/S3 matches: 7,638,365
Average matches per S1: 3.4613


In [15]:
match_distribution_df = pd.DataFrame([
    {
        "matches_per_S1": n,
        "S1_entities": count,
        "percentage": round(count / gt_rows * 100, 3)
    }
    for n, count in sorted(match_distribution.items())
])

display(match_distribution_df)


,matches_per_S1,S1_entities,percentage
0,0,123247,5.585
1,1,119157,5.399
2,2,375212,17.002
3,3,530841,24.055
4,4,484115,21.937
5,5,321957,14.589
6,6,164868,7.471
7,7,63968,2.899
8,8,18680,0.846
9,9,4205,0.191


In [16]:
zero_matches = match_distribution.get(0, 0)
single_match = match_distribution.get(1, 0)
multiple_matches = sum(
    count for n, count in match_distribution.items() if n >= 2
)

print("MATCH STRUCTURE")
print("=" * 70)
print("Zero matches:", f"{zero_matches:,}", f"({zero_matches / gt_rows * 100:.2f}%)")
print("Exactly one match:", f"{single_match:,}", f"({single_match / gt_rows * 100:.2f}%)")
print("Multiple matches:", f"{multiple_matches:,}", f"({multiple_matches / gt_rows * 100:.2f}%)")
print("Has at least one match:", f"{gt_rows - zero_matches:,}",
      f"({(gt_rows - zero_matches) / gt_rows * 100:.2f}%)")


MATCH STRUCTURE
Zero matches: 123,247 (5.58%)
Exactly one match: 119,157 (5.40%)
Multiple matches: 1,964,417 (89.02%)
Has at least one match: 2,083,574 (94.42%)


In [17]:
source_distribution_df = pd.DataFrame([
    {
        "source": source,
        "matches": count,
        "percentage": round(count / total_matches * 100, 3)
    }
    for source, count in source_distribution.items()
])

display(source_distribution_df)


,source,matches,percentage
0,S2,3693619,48.356
1,S3,3944746,51.644


In [18]:
invalid_gt_ids = []

for chunk in pd.read_csv(
    ground_truth,
    sep="\t",
    chunksize=200_000
):
    for value in chunk["matched_entity_ids"].fillna(""):
        if not str(value).strip():
            continue

        ids = [x.strip() for x in str(value).split(",") if x.strip()]

        for entity_id in ids:
            if not (
                entity_id.startswith("S2-")
                or entity_id.startswith("S3-")
            ):
                invalid_gt_ids.append(entity_id)

print("Invalid S2/S3 IDs found:", len(invalid_gt_ids))

if invalid_gt_ids:
    print("Examples:", invalid_gt_ids[:20])


Invalid S2/S3 IDs found: 0


In [19]:
TEXT_SAMPLE_ROWS = 200_000

text_samples = {}

for name in [
    "train_source1",
    "train_source2",
    "train_source3"
]:
    print(f"Loading text sample: {name}")

    text_samples[name] = pd.read_csv(
        files[name],
        sep="\t",
        nrows=TEXT_SAMPLE_ROWS,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    )

    print(f"{name}: {len(text_samples[name]):,} rows")


Loading text sample: train_source1
train_source1: 200,000 rows
Loading text sample: train_source2
train_source2: 200,000 rows
Loading text sample: train_source3
train_source3: 200,000 rows


In [20]:
def text_length_stats(df, column):
    values = df[column].fillna("").astype(str)
    lengths = values.str.len()

    return {
        "column": column,
        "missing": int(df[column].isna().sum()),
        "min": int(lengths.min()),
        "mean": round(lengths.mean(), 2),
        "median": round(lengths.median(), 2),
        "p95": round(lengths.quantile(0.95), 2),
        "max": int(lengths.max())
    }

length_results = []

for name, df in text_samples.items():
    for column in ["business_name", "business_address"]:
        result = text_length_stats(df, column)
        result["dataset"] = name
        length_results.append(result)

length_df = pd.DataFrame(length_results)

display(
    length_df[
        [
            "dataset",
            "column",
            "missing",
            "min",
            "mean",
            "median",
            "p95",
            "max"
        ]
    ]
)


,dataset,column,missing,min,mean,median,p95,max
0,train_source1,business_name,0,3,24.06,24.0,37.0,81
1,train_source1,business_address,0,13,52.07,41.0,103.0,222
2,train_source2,business_name,0,2,25.09,25.0,40.0,104
3,train_source2,business_address,6641,0,46.34,37.0,96.0,204
4,train_source3,business_name,1,0,25.21,25.0,42.0,81
5,train_source3,business_address,6631,0,46.78,42.0,91.0,200


In [21]:
def multilingual_stats(series):
    values = series.fillna("").astype(str)

    non_ascii = values.str.contains(
        r"[^\x00-\x7F]",
        regex=True
    )

    return {
        "total": len(values),
        "non_ascii": int(non_ascii.sum()),
        "non_ascii_pct": round(non_ascii.mean() * 100, 3)
    }

multilingual_results = []

for name, df in text_samples.items():
    for column in ["business_name", "business_address"]:
        stats = multilingual_stats(df[column])
        stats["dataset"] = name
        stats["column"] = column
        multilingual_results.append(stats)

multilingual_df = pd.DataFrame(multilingual_results)
display(multilingual_df)


,total,non_ascii,non_ascii_pct,dataset,column
0,200000,0,0.000,train_source1,business_name
1,200000,43,0.022,train_source1,business_address
2,200000,30130,15.065,train_source2,business_name
3,200000,19012,9.506,train_source2,business_address
4,200000,23214,11.607,train_source3,business_name
5,200000,18188,9.094,train_source3,business_address


In [22]:
def text_pattern_stats(series):
    values = series.fillna("").astype(str)

    return {
        "missing": int((values == "").sum()),
        "contains_digits": int(values.str.contains(r"\d", regex=True).sum()),
        "contains_punctuation": int(values.str.contains(r"[^\w\s]", regex=True).sum()),
        "contains_ampersand": int(values.str.contains("&", regex=False).sum()),
        "contains_dot": int(values.str.contains(r"\.", regex=True).sum()),
        "contains_comma": int(values.str.contains(",", regex=False).sum())
    }

pattern_results = []

for name, df in text_samples.items():
    for column in ["business_name", "business_address"]:
        stats = text_pattern_stats(df[column])
        stats["dataset"] = name
        stats["column"] = column
        pattern_results.append(stats)

pattern_df = pd.DataFrame(pattern_results)
display(pattern_df)


,missing,contains_digits,contains_punctuation,contains_ampersand,contains_dot,contains_comma,dataset,column
0,0,3212,41591,10099,13480,15283,train_source1,business_name
1,0,193067,200000,1761,29643,200000,train_source1,business_address
2,0,10278,85293,8275,23941,13647,train_source2,business_name
3,6641,181233,193359,1615,31336,193359,train_source2,business_address
4,1,10164,80317,8307,23225,13589,train_source3,business_name
5,6631,181614,193369,1357,29014,193369,train_source3,business_address


In [23]:
print("BUSINESS NAME EXAMPLES")
print("=" * 70)

for name, df in text_samples.items():
    print(f"\n{name}")
    display(
        df[
            ["entity_id", "business_name", "country"]
        ].sample(
            min(20, len(df)),
            random_state=42
        )
    )


BUSINESS NAME EXAMPLES

train_source1


,entity_id,business_name,country
119737,S1-313131618,Martin and King Inc,US
72272,S1-580664796,Benally Hancock Group,US
158154,S1-798614098,Blue Project III,US
65426,S1-783901757,Heritage League,US
30074,S1-528308886,"Drayance Aviation, Inc",US
23677,S1-340746308,Shilpi Consultancy Company,India
134858,S1-975237208,"Gladys Vance Great, Inc",US
176418,S1-739671275,Nova Fabrics (India) Company,India
132467,S1-648865832,Joint East Public Limited,India
4082,S1-40587009,Reeves Institute of Technology,US



train_source2


,entity_id,business_name,country
119737,S2-847356727,LP FLORES VALLEY HIGHLAND MICROELECTRONICS,US
72272,S2-248937598,Scholarship ÍI 6uild,US
158154,S2-58604769,"Oneal, Arnold and Neal",US
65426,S2-99285537,halterman deli,US
30074,S2-849630940,Joella Braga Bynordic Ltd,US
23677,S2-404576751,Hitech Health (India) Limited,India
134858,S2-119544367,Buchanan's Federal Media LLC,US
176418,S2-471268796,"Brandon, Tippins & Smith",US
132467,S2-681856622,Regional Council of Village Of Iola Inc,US
4082,S2-320320411,Devika Grkroa Limited,India



train_source3


,entity_id,business_name,country
119737,S3-266771511,DZ Beacon Bitwise [LLC],US
72272,S3-83735803,Rangareddi Ínnovations LLP,India
158154,S3-58290818,WRIGHT SÉCURITIES,US
65426,S3-32679267,Laxmi Cónsulting Pvt Ltd,India
30074,S3-970259864,#kusumestate,India
23677,S3-36748240,Teasel Mánagement,India
134858,S3-9742546,"0dilia Fay Grayscale,",US
176418,S3-668590261,Housing Committee,US
132467,S3-361390317,martinez maywood llc,US
4082,S3-663652167,anand (india) hotels private limited,India


In [24]:
print("ADDRESS EXAMPLES")
print("=" * 70)

for name, df in text_samples.items():
    print(f"\n{name}")
    display(
        df[
            ["entity_id", "business_address", "country"]
        ].sample(
            min(20, len(df)),
            random_state=42
        )
    )


ADDRESS EXAMPLES

train_source1


,entity_id,business_address,country
119737,S1-313131618,"602 Ida Street, Stayton, OR",US
72272,S1-580664796,"39861 Bosque Road, Hempstead, TX",US
158154,S1-798614098,"785 Parklin Avenue, Sacramento, CA",US
65426,S1-783901757,"124 Grant Boulevard, West Seneca, NY",US
30074,S1-528308886,"1650 Sunset Park Drive, Nolensville, TN",US
23677,S1-340746308,"C/O Amshya Fulji Padvi At Post Koyalivihir, Tal-Akkalkuwa, Dist-Nandurbar, Nandurbar, Maharashtra",India
134858,S1-975237208,"4848 Overhill Avenue, Norridge, IL",US
176418,S1-739671275,"Plot No. 1042, Laxmi Sagar, Bhubaneswar, Khordha, Bhubaneswar, Khordha, Orissa",India
132467,S1-648865832,"10/58, Ground Floor, Kirti Nagar Industrial Area, New Delhi, Delhi, West Delhi, Delhi",India
4082,S1-40587009,"5221 Wickberg Road, Snowflake, AZ",US



train_source2


,entity_id,business_address,country
119737,S2-847356727,"759 CRESCENT STREET, BROCKTON, MA",US
72272,S2-248937598,"WA, <NULL>, PUYALLUP, 916 133RD ST",US
158154,S2-58604769,"774-D CORBIN CREST TRL, LA PORTE, TX",US
65426,S2-99285537,"618 CENTRAL AVE, DUNKIRK, NY",US
30074,S2-849630940,"N/A, WI, N91W17069 APPLE TREE CT, MENOMONEE FALLS",US
23677,S2-404576751,"C 1/25/5 FF HUKUM CHAND APPT, NAND VIHAR SEC 16 A, NR DWARKA METRO STATION, NEW DELHI, Delhi",India
134858,S2-119544367,"4834 AVOCET CT, OH, DAYTON",US
176418,S2-471268796,"900 57 LAKESIDE DRIVE, AMARILLO CITY, TX",US
132467,S2-681856622,"115 MAIN ST, IOLA, WI",US
4082,S2-320320411,"H.NO.4-1-319/2/A , PLOT NO.2, ROAD NO.1, NEW MAMATHA NAGAR, NAGOLE, HYDERABAD, Telangana",India



train_source3


,entity_id,business_address,country
119737,S3-266771511,"0083 Lewis Road, Proctor, Arkansas",US
72272,S3-83735803,"208, Rangareddi, Rangareddy, Andhra Pradesh",India
158154,S3-58290818,"6507 Sunset Blvd, Macy, Indiana",US
65426,S3-32679267,"New Delhi, North Delhi, 250 Kalyan Vihar, DL",India
30074,S3-970259864,"Pune City, Pune, Bunglow No 3, MH, Sr 135/136, Mantri Kishor Park, Wpump",India
23677,S3-36748240,"Plot 535 G-1702, Panchshil Towers, Gat No 1277 To 1283, Haveli, Pune, MH",India
134858,S3-9742546,"163 Woodward Ave, Newport, North Carolina",US
176418,S3-668590261,"11928 1/2 Paseo Del Rey Dr, City Of El Paso, Texas",US
132467,S3-361390317,"4516 Glines Avenue, Orcutt, California",US
4082,S3-663652167,"Front Of Rajib Market College Road, Nayagarh, OD",India


In [25]:
# Inspect a small set of ground-truth examples.
# We load only the required S1/S2/S3 rows for the first 10 ground-truth records.

gt_examples = pd.read_csv(
    ground_truth,
    sep="\t",
    nrows=10
)

s1_examples = pd.read_csv(
    train_s1,
    sep="\t",
    dtype=str
)

s2_examples = pd.read_csv(
    train_s2,
    sep="\t",
    dtype=str
)

s3_examples = pd.read_csv(
    train_s3,
    sep="\t",
    dtype=str
)

s1_lookup = s1_examples.set_index("entity_id")
s2_lookup = s2_examples.set_index("entity_id")
s3_lookup = s3_examples.set_index("entity_id")

for _, row in gt_examples.iterrows():
    s1_id = row["source1_entity_id"]
    match_string = str(row["matched_entity_ids"])

    print("\n" + "=" * 100)
    print("S1:", s1_id)

    if s1_id in s1_lookup.index:
        display(
            pd.DataFrame([s1_lookup.loc[s1_id]])
        )

    matched_ids = [
        x.strip()
        for x in match_string.split(",")
        if x.strip()
    ]

    matched_rows = []

    for entity_id in matched_ids:
        if entity_id.startswith("S2-") and entity_id in s2_lookup.index:
            r = s2_lookup.loc[entity_id].copy()
            r["matched_source"] = "S2"
            matched_rows.append(r)

        elif entity_id.startswith("S3-") and entity_id in s3_lookup.index:
            r = s3_lookup.loc[entity_id].copy()
            r["matched_source"] = "S3"
            matched_rows.append(r)

    if matched_rows:
        display(pd.DataFrame(matched_rows))

del s1_examples, s2_examples, s3_examples
del s1_lookup, s2_lookup, s3_lookup
del gt_examples
gc.collect()



S1: S1-965667


,business_name,business_address,country
S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",US


,business_name,business_address,country,matched_source
S2-681193310,Maure Wilblims Colombier Inc,NaN,US,S2
S2-743505751,Maure Williams Colombier,NaN,US,S2
S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US,S3
S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York",US,S3
S3-860443364,Maure Williams Inc Center,NaN,US,S3



S1: S1-55344266


,business_name,business_address,country
S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",India


...

,business_name,business_address,country,matched_source
S2-648035184,Hendricks and Flowers Inc,"CT, SLEEPY HOLLOW DRIVE, DANBURY",US,S2
S3-588502663,Hendricks and Inc Flowers,NaN,US,S3



S1: S1-789009573


,business_name,business_address,country
S1-789009573,Hotel Enterprises Limited,"Wz-187C Shop No.13, 14 Kh. No.47 S/F. Vikaspuri Budhela Village Behind Oxford School, Delhi, West Delhi, Delhi",India


,business_name,business_address,country,matched_source
S2-383871912,होटल एंटरप्राइजेज लिमिटेड,"WZ-187C SHOP NO.13, DELHI, WEST DELHI, Delhi",India,S2
S3-74481402,Hotel Limited Services,"Block B-517 Wz-187c Shop No.13, Divreportingcircle, West Delhi, DL",India,S3
S3-576451439,Hotel Énterprises Limited,"Block B-517 Wz-187c Shop No.13, South West Delhi, Delhi, DL",India,S3


0

## Profiling Summary

At the end of this notebook, review:

1. Total rows in S1/S2/S3.
2. Country distributions.
3. Missing business names and addresses.
4. Duplicate entity IDs.
5. Percentage of S1 entities with zero, one, or multiple matches.
6. S2 versus S3 match proportions.
7. Name/address length distributions.
8. Multilingual/non-ASCII text frequency.
9. Real examples of noisy entity matches.

These statistics will be used in the next phase to design:
- multilingual-safe normalization
- multi-pass blocking
- candidate-pair generation
- pairwise similarity features
- entity-level matching and thresholding


In [26]:
print("COUNTRY DISTRIBUTION")
print("=" * 70)

for name, path in files.items():
    if name not in [
        "train_source1",
        "train_source2",
        "train_source3"
    ]:
        continue

    counts = Counter()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["country"],
        chunksize=200_000
    ):
        counts.update(
            chunk["country"]
            .fillna("<MISSING>")
            .astype(str)
            .str.strip()
        )

    print(f"\n{name}")
    print("-" * 50)

    total = sum(counts.values())

    for country, count in counts.most_common(20):
        print(
            f"{country:20s} "
            f"{count:>10,} "
            f"({count / total * 100:6.2f}%)"
        )

COUNTRY DISTRIBUTION

train_source1
--------------------------------------------------
US                    1,323,633 ( 59.98%)
India                   883,188 ( 40.02%)

train_source2
--------------------------------------------------
US                    3,016,817 ( 59.92%)
India                 2,017,799 ( 40.08%)

train_source3
--------------------------------------------------
US                    3,170,056 ( 59.98%)
India                 2,115,547 ( 40.02%)


In [27]:
def load_country_maps(path):
    result = {}

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country"],
        dtype=str,
        chunksize=200_000
    ):
        for entity_id, country in zip(
            chunk["entity_id"],
            chunk["country"]
        ):
            result[entity_id] = (
                "" if pd.isna(country)
                else str(country).strip().lower()
            )

    return result


print("Loading country maps...")

s1_country = load_country_maps(train_s1)
s2_country = load_country_maps(train_s2)
s3_country = load_country_maps(train_s3)

print("Loaded:")
print("S1:", len(s1_country))
print("S2:", len(s2_country))
print("S3:", len(s3_country))

Loading country maps...
Loaded:
S1: 2206821
S2: 5034616
S3: 5285603


In [28]:
same_country = 0
different_country = 0
missing_country = 0
total = 0

for chunk in pd.read_csv(
    ground_truth,
    sep="\t",
    chunksize=200_000
):

    for _, row in chunk.iterrows():

        s1_id = row["source1_entity_id"]

        s1c = s1_country.get(s1_id, "")

        value = "" if pd.isna(row["matched_entity_ids"]) else str(
            row["matched_entity_ids"]
        ).strip()

        if not value:
            continue

        for entity_id in value.split(","):

            entity_id = entity_id.strip()

            if entity_id.startswith("S2-"):
                mc = s2_country.get(entity_id, "")
            elif entity_id.startswith("S3-"):
                mc = s3_country.get(entity_id, "")
            else:
                continue

            total += 1

            if not s1c or not mc:
                missing_country += 1
            elif s1c == mc:
                same_country += 1
            else:
                different_country += 1

print("\nGROUND-TRUTH COUNTRY CONSISTENCY")
print("=" * 70)

print("Total matches:", f"{total:,}")
print(
    "Same country:",
    f"{same_country:,}",
    f"({same_country / total * 100:.3f}%)"
)
print(
    "Different country:",
    f"{different_country:,}",
    f"({different_country / total * 100:.3f}%)"
)
print(
    "Missing country:",
    f"{missing_country:,}",
    f"({missing_country / total * 100:.3f}%)"
)


GROUND-TRUTH COUNTRY CONSISTENCY
Total matches: 7,638,365
Same country: 7,638,365 (100.000%)
Different country: 0 (0.000%)
Missing country: 0 (0.000%)


In [29]:
import re
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""

    s = str(s).lower().strip()

    # Unicode normalization
    s = unicodedata.normalize("NFKC", s)

    # Keep letters/numbers, convert punctuation to spaces
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)

    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [30]:
def profile_normalized_names(path, source_name):
    counts = Counter()
    total = 0
    empty = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["business_name"],
        dtype=str,
        chunksize=200_000
    ):
        names = chunk["business_name"].map(normalize_text)

        empty += (names == "").sum()
        total += len(names)
        counts.update(names)

    unique = len(counts)
    duplicate_records = total - unique

    print(f"\n{source_name}")
    print("=" * 60)
    print(f"Total records:       {total:,}")
    print(f"Unique norm names:   {unique:,}")
    print(f"Duplicate records:   {duplicate_records:,}")
    print(f"Empty normalized:    {empty:,}")

    print("\nMost common normalized names:")
    for name, count in counts.most_common(15):
        print(f"{count:>8,}  {name[:100]}")

    return counts


s1_name_counts = profile_normalized_names(
    train_s1,
    "TRAIN SOURCE 1"
)

s2_name_counts = profile_normalized_names(
    train_s2,
    "TRAIN SOURCE 2"
)

s3_name_counts = profile_normalized_names(
    train_s3,
    "TRAIN SOURCE 3"
)


TRAIN SOURCE 1
Total records:       2,206,821
Unique norm names:   1,520,684
Duplicate records:   686,137
Empty normalized:    0

Most common normalized names:
     253  primary care group
     251  ear nose throat group
     222  pediatric group
     220  womens health group
     218  physical therapy group
     216  pediatric dental group
     215  behavioral health group
     209  chiropractic group
     208  orthopedic group
     207  eye group
     205  dental group
     204  family group
     202  dermatology group
     199  vision group
     198  pediatric dentistry group

TRAIN SOURCE 2
Total records:       5,034,616
Unique norm names:   4,025,057
Duplicate records:   1,009,559
Empty normalized:    2

Most common normalized names:
     526  physical therapy
     518  primary care
     483  womens health
     469  urgent care
     468  behavioral health
     468  internal medicine
     468  pediatric dental
     453  pediatric dentistry
     369  foot ankle
     362  ear nose t

## Phase 2: Multi-Representation Text Normalization

The challenge contains noisy business names and addresses, including abbreviations, punctuation changes, typos, transliterations, missing address components, and reordered components. Instead of replacing the raw text with one aggressive normalization, we keep multiple representations for blocking and later feature engineering.

Representations created here:
- `*_name_norm`: conservative normalized business name
- `*_name_compact`: punctuation/space-free name for compact exact matching
- `*_name_tokens`: normalized token set
- `*_name_token_sorted`: order-insensitive representation
- `*_address_norm`: conservative normalized address
- `*_address_compact`: compact address representation
- `*_address_tokens`: normalized address token set


In [31]:
# ============================================================
# Phase 2A — Multi-Representation Text Normalization
# ============================================================

LEGAL_SUFFIXES = {
    "limited", "ltd", "llp", "llc", "inc", "incorporated",
    "corporation", "corp", "company", "co", "private", "pvt"
}

NAME_ABBREVIATIONS = {
    "&": "and",
}

ADDRESS_ABBREVIATIONS = {
    "rd": "road", "rd.": "road",
    "st": "street", "st.": "street",
    "ste": "suite", "apt": "apartment",
    "hwy": "highway", "blvd": "boulevard",
    "ave": "avenue", "av": "avenue",
    "ln": "lane", "dr": "drive", "ct": "court",
    "pkwy": "parkway", "pl": "place", "sq": "square",
}


def _unicode_basic(s):
    """Unicode-safe lowercase + punctuation-to-space normalization."""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", str(s)).lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_name_views(s):
    """Return conservative and order-insensitive business-name views."""
    base = _unicode_basic(s)
    tokens = base.split()

    # Keep legal suffixes in the conservative representation.
    norm = " ".join(tokens)

    # Separate representation without common legal suffixes.
    core_tokens = [t for t in tokens if t not in LEGAL_SUFFIXES]
    core = " ".join(core_tokens)

    # Order-insensitive view helps with token transpositions.
    token_sorted = " ".join(sorted(core_tokens))

    # Compact representation is useful for exact blocking after normalization.
    compact = re.sub(r"[^\w]", "", core, flags=re.UNICODE)

    return norm, compact, " ".join(core_tokens), token_sorted


def normalize_address_views(s):
    """Return conservative, abbreviation-normalized address views."""
    base = _unicode_basic(s)
    tokens = base.split()

    expanded = [ADDRESS_ABBREVIATIONS.get(t, t) for t in tokens]
    norm = " ".join(expanded)
    compact = re.sub(r"[^\w]", "", norm, flags=re.UNICODE)
    token_sorted = " ".join(sorted(set(expanded)))

    return norm, compact, " ".join(expanded), token_sorted


# Quick sanity checks before applying the functions to the full data.
normalization_examples = pd.DataFrame({
    "raw_name": [
        "ABC Corp.",
        "ABC Corporation",
        "A.B.C. Pvt Ltd",
        "The Royal Cafe & Restaurant",
    ],
    "raw_address": [
        "12 MG Rd.",
        "12, MG Road",
        "Shop 4, Main St.",
        "Near SBI ATM, 18 Park Ave",
    ]
})

name_views = normalization_examples["raw_name"].map(normalize_name_views)
address_views = normalization_examples["raw_address"].map(normalize_address_views)

normalization_examples["name_norm"] = name_views.str[0]
normalization_examples["name_compact"] = name_views.str[1]
normalization_examples["name_token_sorted"] = name_views.str[3]
normalization_examples["address_norm"] = address_views.str[0]
normalization_examples["address_compact"] = address_views.str[1]

print("NORMALIZATION SANITY CHECK")
display(normalization_examples)


NORMALIZATION SANITY CHECK


,raw_name,raw_address,name_norm,name_compact,name_token_sorted,address_norm,address_compact
0,ABC Corp.,12 MG Rd.,abc corp,abc,abc,12 mg road,12mgroad
1,ABC Corporation,"12, MG Road",abc corporation,abc,abc,12 mg road,12mgroad
2,A.B.C. Pvt Ltd,"Shop 4, Main St.",a b c pvt ltd,abc,a b c,shop 4 main street,shop4mainstreet
3,The Royal Cafe & Restaurant,"Near SBI ATM, 18 Park Ave",the royal cafe and restaurant,theroyalcafeandrestaurant,and cafe restaurant royal the,near sbi atm 18 park avenue,nearsbiatm18parkavenue


## Phase 2B: Build Normalized Columns for Training Data

For the next stages we materialize the normalized representations for the training sources. These columns will be used for blocking and pairwise feature generation. Raw columns remain untouched so that later models can compare raw and normalized signals.


> **Performance note:** Training files are normalized in chunks so progress is visible and memory usage is controlled.


In [32]:
# ============================================================
# Phase 2B — Apply normalization to training sources
# ============================================================

# The path variables (train_s1, train_s2, train_s3) point to TSV files.
# Phase 2 needs DataFrames, so we load and normalize them in chunks.
# Chunking gives visible progress and avoids one long apparently-frozen operation.

NORMALIZATION_CHUNK_SIZE = 50_000

def add_normalized_columns_from_path(path, source_name, chunksize=NORMALIZATION_CHUNK_SIZE):
    columns = [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]

    chunks = []
    total_rows = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            usecols=columns,
            dtype=str,
            chunksize=chunksize
        ),
        start=1
    ):
        name_views = chunk["business_name"].fillna("").map(normalize_name_views)
        address_views = chunk["business_address"].fillna("").map(normalize_address_views)

        chunk["name_norm"] = name_views.str[0]
        chunk["name_compact"] = name_views.str[1]
        chunk["name_tokens"] = name_views.str[2]
        chunk["name_token_sorted"] = name_views.str[3]

        chunk["address_norm"] = address_views.str[0]
        chunk["address_compact"] = address_views.str[1]
        chunk["address_tokens"] = address_views.str[2]
        chunk["address_token_sorted"] = address_views.str[3]

        chunks.append(chunk)
        total_rows += len(chunk)

        print(
            f"{source_name}: processed {total_rows:,} rows",
            flush=True
        )

    result = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    print(f"{source_name}: normalization complete -> {len(result):,} rows")
    return result


train_s1_norm = add_normalized_columns_from_path(train_s1, "S1")
train_s2_norm = add_normalized_columns_from_path(train_s2, "S2")
train_s3_norm = add_normalized_columns_from_path(train_s3, "S3")

print("\nNormalized training shapes:")
print("S1:", train_s1_norm.shape)
print("S2:", train_s2_norm.shape)
print("S3:", train_s3_norm.shape)

display(
    train_s1_norm[
        ["entity_id", "business_name", "name_norm", "name_compact",
         "business_address", "address_norm", "country"]
    ].head(10)
)


S1: processed 50,000 rows
S1: processed 100,000 rows
S1: processed 150,000 rows
S1: processed 200,000 rows
S1: processed 250,000 rows
S1: processed 300,000 rows
S1: processed 350,000 rows
S1: processed 400,000 rows
S1: processed 450,000 rows
S1: processed 500,000 rows
S1: processed 550,000 rows
S1: processed 600,000 rows
S1: processed 650,000 rows
S1: processed 700,000 rows
S1: processed 750,000 rows
S1: processed 800,000 rows
S1: processed 850,000 rows
S1: processed 900,000 rows
S1: processed 950,000 rows
S1: processed 1,000,000 rows
S1: processed 1,050,000 rows
S1: processed 1,100,000 rows
S1: processed 1,150,000 rows
S1: processed 1,200,000 rows
S1: processed 1,250,000 rows
S1: processed 1,300,000 rows
S1: processed 1,350,000 rows
S1: processed 1,400,000 rows
S1: processed 1,450,000 rows
S1: processed 1,500,000 rows
S1: processed 1,550,000 rows
S1: processed 1,600,000 rows
S1: processed 1,650,000 rows
S1: processed 1,700,000 rows
S1: processed 1,750,000 rows
S1: processed 1,800,000 

,entity_id,business_name,name_norm,name_compact,business_address,address_norm,country
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,oreleesbarbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,S1-773889195,Prime Money,prime money,primemoney,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,S1-377745466,B+ Retail Inc,b retail inc,bretail,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,S1-133037285,Christ Chapel,christ chapel,christchapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,S1-755362802,Prabhav Business Center,prabhav business center,prabhavbusinesscenter,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",797 lake town block a kolkata howrah west bengal,India
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,customwealthservices,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,consultingnyasanursing,"2505, Tower 1, Oakwood, Runwal Greens, Mulund Goreagon Link Road, Near Fortis Hospital, Bhandup West, Mumbai, Mahara...",2505 tower 1 oakwood runwal greens mulund goreagon link road near fortis hospital bhandup west mumbai maharashtra,India
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,nexusanchorrain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,moorebitwise,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,dermatologygreenmedicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of Pewaukee, WI",294 meadowcreek drive unit unit 2 village of pewaukee wi,US


## Phase 2C: Normalization Quality Checks

Before blocking, inspect how much normalization collapses distinct records. This is important because an exact normalized key is useful for high-precision blocking, but a very common key should not be treated as an automatic match.


In [33]:
# ============================================================
# Phase 2C — Normalization quality checks
# ============================================================

def normalization_quality(df, source_name):
    result = {
        "source": source_name,
        "rows": len(df),
        "empty_name_norm": int((df["name_norm"] == "").sum()),
        "unique_name_norm": int(df["name_norm"].nunique()),
        "unique_name_compact": int(df["name_compact"].nunique()),
        "empty_address_norm": int((df["address_norm"] == "").sum()),
        "unique_address_norm": int(df["address_norm"].nunique()),
    }
    result["name_norm_collision_rate"] = 1 - result["unique_name_norm"] / max(result["rows"], 1)
    return result

normalization_quality_df = pd.DataFrame([
    normalization_quality(train_s1_norm, "S1"),
    normalization_quality(train_s2_norm, "S2"),
    normalization_quality(train_s3_norm, "S3"),
])

display(normalization_quality_df)

print("\nMost common normalized names across each source:")
for source_name, df in [("S1", train_s1_norm), ("S2", train_s2_norm), ("S3", train_s3_norm)]:
    print(f"\n{source_name}")
    display(
        df.loc[df["name_norm"].ne(""), "name_norm"]
          .value_counts()
          .head(15)
          .rename_axis("name_norm")
          .reset_index(name="records")
    )


,source,rows,empty_name_norm,unique_name_norm,unique_name_compact,empty_address_norm,unique_address_norm,name_norm_collision_rate
0,S1,2206821,0,1522166,1358514,0,2130038,0.310245
1,S2,5034616,2,4030167,3430798,168967,4090610,0.199509
2,S3,5285603,13,4286721,3671425,175916,4438318,0.188982



Most common normalized names across each source:

S1


,name_norm,records
0,primary care group,253
1,ear nose and throat group,251
2,pediatric group,222
3,womens health group,220
4,physical therapy group,218
5,pediatric dental group,216
6,behavioral health group,215
7,chiropractic group,209
8,orthopedic group,208
9,eye group,207



S2


,name_norm,records
0,physical therapy,526
1,primary care,518
2,womens health,483
3,urgent care,469
4,behavioral health,468
5,pediatric dental,468
6,internal medicine,468
7,pediatric dentistry,453
8,ear nose and throat,357
9,foot and ankle,356



S3


,name_norm,records
0,primary care,521
1,physical therapy,516
2,pediatric dental,503
3,urgent care,480
4,womens health,469
5,pediatric dentistry,465
6,internal medicine,427
7,behavioral health,427
8,foot and ankle,337
9,pediatric,326
